# QOS Circuit Depth & 2Q Gate Scaling
**Tommaso R. Marena (2026) — Appendix B**

Measures how the number of 2-qubit gates and circuit depth scales with n (number of qubits)
for both the worst-case (dense) WHT decomposition and the K-sparse oracle decomposition.
Produces the scaling figure referenced in the hardware section.

**Produces:**
- `circuit_depth_vs_n.pdf` — 2Q gate count vs n with O(2^n) and O(K log K) fit lines
- `circuit_depth_table.json` — raw gate counts

> **Runtime:** ~5 min (CPU only, no GPU needed).

In [ ]:
import subprocess, sys
for pkg in ['qiskit>=1.1', 'numpy', 'matplotlib',
            'quantum-oracle-sketching[dev] @ git+https://github.com/Tommaso-R-Marena/quantum_oracle_sketching.git']:
    subprocess.run([sys.executable,'-m','pip','install','-q',pkg], capture_output=True)
print('Installed.')

In [ ]:
import os
MOUNT_DRIVE = True
if MOUNT_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/qos_hardware'
else:
    OUTPUT_DIR = '/content/qos_depth'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
import numpy as np, jax.numpy as jnp, json
import matplotlib.pyplot as plt, matplotlib
matplotlib.rcParams.update({'figure.dpi': 150, 'font.size': 10, 'font.family': 'serif'})
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qos.core.oracle_sketch import q_oracle_sketch_boolean

N_RANGE   = list(range(2, 10))   # n=2..9 qubits
M_SAMPLES = 1024
SEED      = 42
rng = np.random.default_rng(SEED)

def count_2q_gates(qc):
    ops = qc.count_ops()
    return sum(v for k, v in ops.items() if k in ('cx','ecr','cz','ccx','mcx'))

def build_circuit_dense(n, M):
    """Dense WHT decomposition: O(2^n) CX gates (worst case, all k non-zero)."""
    N = 2**n
    tt = jnp.array(rng.integers(0,2,size=N), dtype=jnp.float64)
    diag, _ = q_oracle_sketch_boolean(tt, M)
    phases = np.angle(np.array(diag))
    H = np.array([[1,1],[1,-1]])/np.sqrt(2)
    Hn = H.copy()
    for _ in range(n-1): Hn = np.kron(Hn, H)
    wht = (Hn @ phases) / N
    EPS = 1e-8
    qr = QuantumRegister(n); cr = ClassicalRegister(n)
    qc = QuantumCircuit(qr, cr)
    qc.h(qr)
    for k in range(N):
        theta = float(wht[k])
        if abs(theta) < EPS: continue
        bits = [j for j in range(n) if (k >> j) & 1]
        if len(bits) == 0: continue
        elif len(bits) == 1: qc.p(2*theta, qr[bits[0]])
        else:
            tgt = bits[-1]
            for c in bits[:-1]: qc.cx(qr[c], qr[tgt])
            qc.p(2*theta, qr[tgt])
            for c in reversed(bits[:-1]): qc.cx(qr[c], qr[tgt])
    qc.h(qr); qc.measure(qr,cr)
    return qc

def build_circuit_sparse(n, M, K_frac=0.1):
    """K-sparse truth table: only K = K_frac * N entries are 1."""
    N = 2**n
    K = max(1, int(K_frac * N))
    tt_np = np.zeros(N); tt_np[rng.choice(N, K, replace=False)] = 1.0
    tt = jnp.array(tt_np, dtype=jnp.float64)
    diag, _ = q_oracle_sketch_boolean(tt, M)
    phases = np.angle(np.array(diag))
    H = np.array([[1,1],[1,-1]])/np.sqrt(2)
    Hn = H.copy()
    for _ in range(n-1): Hn = np.kron(Hn, H)
    wht = (Hn @ phases) / N
    EPS = 1e-8
    qr = QuantumRegister(n); cr = ClassicalRegister(n)
    qc = QuantumCircuit(qr, cr)
    qc.h(qr)
    for k in range(N):
        theta = float(wht[k])
        if abs(theta) < EPS: continue
        bits = [j for j in range(n) if (k >> j) & 1]
        if len(bits) == 0: continue
        elif len(bits) == 1: qc.p(2*theta, qr[bits[0]])
        else:
            tgt = bits[-1]
            for c in bits[:-1]: qc.cx(qr[c], qr[tgt])
            qc.p(2*theta, qr[tgt])
            for c in reversed(bits[:-1]): qc.cx(qr[c], qr[tgt])
    qc.h(qr); qc.measure(qr,cr)
    return qc, K

# ── Run ─────────────────────────────────────────────────────
table = {}
for n in N_RANGE:
    qc_d = build_circuit_dense(n, M_SAMPLES)
    qc_s, K = build_circuit_sparse(n, M_SAMPLES, K_frac=0.1)
    two_q_dense  = count_2q_gates(qc_d)
    two_q_sparse = count_2q_gates(qc_s)
    depth_dense  = qc_d.depth()
    depth_sparse = qc_s.depth()
    table[n] = {'N': 2**n, 'K': K,
                'two_q_dense':  two_q_dense,  'depth_dense':  depth_dense,
                'two_q_sparse': two_q_sparse, 'depth_sparse': depth_sparse}
    print(f'n={n}: N={2**n}, K={K}, 2Q_dense={two_q_dense}, 2Q_sparse={two_q_sparse}')

with open(os.path.join(OUTPUT_DIR,'circuit_depth_table.json'),'w') as f:
    json.dump(table, f, indent=2)

# ── Plot ─────────────────────────────────────────────────────
ns      = sorted(table.keys())
tq_d    = [table[n]['two_q_dense']  for n in ns]
tq_s    = [table[n]['two_q_sparse'] for n in ns]
Ns      = [2**n for n in ns]
Ks      = [table[n]['K'] for n in ns]

# Fit lines
fit_d = np.polyfit(ns, np.log2(np.array(tq_d)+1), 1)
fit_s = np.polyfit([np.log2(K) for K in Ks], np.log2(np.array(tq_s)+1), 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# Panel 1: 2Q gate count vs n (log scale)
ax1.semilogy(ns, tq_d, 'o-', color='tomato',   lw=2, ms=7, label='Dense oracle (all K)')
ax1.semilogy(ns, tq_s, 's-', color='seagreen',  lw=2, ms=7, label=f'Sparse oracle (K=10% N)')
n_fit = np.linspace(min(ns), max(ns), 100)
ax1.semilogy(n_fit, 2**np.polyval(fit_d, n_fit), 'r--', lw=1, alpha=0.6, label=f'Fit: $O(2^{{{fit_d[0]:.1f}n}})$')
ax1.set_xlabel('Number of qubits $n$')
ax1.set_ylabel('Two-qubit gate count')
ax1.set_title('2Q gate count vs circuit size')
ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)
ax1.set_xticks(ns)

# Panel 2: 2Q gate count vs K (log-log)
ax2.loglog(Ks, tq_s, 's-', color='seagreen', lw=2, ms=7, label='Sparse oracle')
k_fit_range = np.logspace(np.log10(min(Ks)+1), np.log10(max(Ks)), 100)
ax2.loglog(k_fit_range, k_fit_range * np.log2(k_fit_range+2), 'g--', lw=1, alpha=0.7, label='$O(K \log K)$ reference')
ax2.set_xlabel('Sparsity $K$')
ax2.set_ylabel('Two-qubit gate count')
ax2.set_title('Sparse oracle gate count vs $K$')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

plt.suptitle('QOS Oracle Circuit Complexity Scaling', fontsize=11)
plt.tight_layout()
path = os.path.join(OUTPUT_DIR,'circuit_depth_vs_n.pdf')
plt.savefig(path, bbox_inches='tight'); plt.show()
print(f'Saved: {path}')

# LaTeX table
print('\n--- LaTeX table (Appendix B) ---')
print(r'\begin{tabular}{cccccc}')
print(r'\toprule')
print(r'$n$ & $N$ & $K$ & 2Q (dense) & 2Q (sparse) & Reduction \\ ')
print(r'\midrule')
for n in ns:
    t = table[n]
    red = t['two_q_dense'] / max(t['two_q_sparse'], 1)
    print(f"{n} & {t['N']} & {t['K']} & {t['two_q_dense']} & {t['two_q_sparse']} & {red:.1f}x \\\\")
print(r'\bottomrule'); print(r'\end{tabular}')